In [1]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

# Configurações
headers = {"User-Agent": "Mozilla/5.0"}
limite = 50
pagina = 0
todos_produtos = []

while True:
    url = f"https://www.copafer.com.br/api/catalog_system/pub/products/search/pisos-e-revestimentos?O=OrderByReleaseDateDESC&_from={pagina * limite}&_to={(pagina + 1) * limite - 1}"
    print(f"Coletando página {pagina + 1}...")

    resposta = requests.get(url, headers=headers)
    if resposta.status_code not in [200, 206]:
        print(f"Erro ao acessar a API: {resposta.status_code}")
        break

    produtos = resposta.json()
    if not produtos:
        print("Nenhum produto encontrado. Fim da coleta.")
        break

    for produto in produtos:
        try:
            nome = produto.get("productName", "")
            raw_link = produto.get("link", "")
            link = raw_link if raw_link.startswith("http") else "https://www.copafer.com.br" + raw_link
            codigo = produto.get("productReference", "")
            preco_promocional = produto["items"][0]["sellers"][0]["commertialOffer"]["Price"]
            preco_original = produto["items"][0]["sellers"][0]["commertialOffer"]["ListPrice"]
            preco_m2 = round(preco_promocional / produto["items"][0].get("unitMultiplier", 1), 2)

            # Descrição limpa com BeautifulSoup
            descricao_html = produto.get("description", "")
            descricao = BeautifulSoup(descricao_html, "html.parser").get_text(separator="\n").strip()

            especificacoes = produto.get("productSpecifications", "")
            imagem = produto["items"][0]["images"][0]["imageUrl"]
            loja = "Copafer"

            todos_produtos.append({
                "Código": codigo,
                "Nome": nome,
                "Preço Promocional": preco_promocional,
                "Preço Original": preco_original,
                "Preço por m²": preco_m2,
                "Especificações": especificacoes,
                "Descrição": descricao,
                "Imagem": imagem,
                "URL": link,
                "Loja": loja
            })

        except Exception as e:
            print(f"Erro ao processar produto: {e}")

    pagina += 1
    time.sleep(0.5)

# Salvar CSV
df = pd.DataFrame(todos_produtos)
df.to_csv("produtos_pisos_revestimentos_copafer.csv", index=False, encoding="utf-8-sig")
print(f"Total de produtos salvos: {len(df)}")


Coletando página 1...
Coletando página 2...
Coletando página 3...
Coletando página 4...
Coletando página 5...
Coletando página 6...
Coletando página 7...
Coletando página 8...
Coletando página 9...
Coletando página 10...
Coletando página 11...
Coletando página 12...
Coletando página 13...
Coletando página 14...
Nenhum produto encontrado. Fim da coleta.
Total de produtos salvos: 633
